# **Get Legislator Information (Plural)**

This notebook extracts and processes information about **U.S. state current legislators** from YAML files available in the OpenStates dataset. It converts these YAML files into a single, structured DataFrame, ensures all identifier schemes are included, and saves the result as a CSV file.

In [1]:
import pandas as pd                     # For data manipulation
import json                             # For file handling
import os                               # For file handling
#!pip install pyyaml
import yaml                             # To read .yml files
from collections import defaultdict     # To simplify missing key handling

## **Define Source Folder and List YAML Files**

The dataset consists of multiple .yml files located in a local folder that mirrors the OpenStates “people” repository.
Here, we define the folder path, list all files that end with .yml, and print them to verify that the data has been loaded correctly.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Colab Version Importation**

In [3]:
!git clone https://github.com/openstates/people.git

Cloning into 'people'...
remote: Enumerating objects: 342013, done.
remote: Counting objects: 100% (1755/1755), done.
remote: Compressing objects: 100% (1024/1024), done.
remote: Total 342013 (delta 1358), reused 779 (delta 731), pack-reused 340258 (from 3)
Receiving objects: 100% (342013/342013), 97.83 MiB | 20.81 MiB/s, done.
Resolving deltas: 100% (280071/280071), done.
Updating files: 100% (25767/25767), done.


In [4]:
folder_path = "/content/people/data/us/legislature"
files = [f for f in os.listdir(folder_path) if f.endswith(".yml")]
print(len(files), "YAML files found")

537 YAML files found


### **Local Version Importation**

In [5]:
# clone repository from github at https://github.com/openstates/people.git
#folder_path = "data/us/legislature"
#files = [f for f in os.listdir(folder_path) if f.endswith(".yml")]
# print the files
#print(files)

## **Initialize Data Structures**
We initialize two key structures:

- `rows`: a list that will hold each legislator’s information as a dictionary.
- `all_schemes`: a set to collect all possible identifier schemes across files (e.g., bioguide, wikidata, votesmart, etc.).

In [6]:
# Convert all the yaml files to a dataframe
rows = []

# First pass: gather all possible identifier schemes
all_schemes = set()

## **First Pass — Gather Identifier Schemes**

During this first loop through all YAML files:

- We open each file safely using UTF-8 encoding.

- Extract the section "other_identifiers" if it exists.

- Collect each unique "scheme" value (these are different ID systems used across legislators).

This ensures that we know **all possible identifier types** before building the DataFrame.

In [7]:
for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
        for id_obj in data.get("other_identifiers", []):
            all_schemes.add(id_obj.get("scheme"))

## **Second Pass — Build Legislator Records**
Now that all identifier schemes are known, we process each YAML file to extract structured information:

- Basic demographic data (name, gender, birth date)

- Role information (district, chamber, start/end dates)

- Party affiliation

- Contact information and social media IDs

- Other identifiers (bioguide, wikidata, etc.)

Each record is stored as a dictionary in `rows`.

In [8]:
# Now process each file with all identifier columns
for file in files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

        # Prepare default row
        row = defaultdict(lambda: None)

        # Basic fields
        row["id"] = data.get("id")
        row["name"] = data.get("name")
        row["given_name"] = data.get("given_name")
        row["family_name"] = data.get("family_name")
        row["birth_date"] = data.get("birth_date")
        row["gender"] = data.get("gender")
        row["email"] = data.get("email")
        row["image"] = data.get("image")
        row["party"] = data.get("party", [{}])[0].get("name")

        # Roles
        role = data.get("roles", [{}])[-1]
        row["role_type"] = role.get("type")
        row["district"] = role.get("district")
        row["role_start_date"] = role.get("start_date")
        row["role_end_date"] = role.get("end_date")

        # Social media (if available)
        for k, v in data.get("ids", {}).items():
            row[f"social_{k}"] = v

        # Other identifiers
        for id_obj in data.get("other_identifiers", []):
            scheme = id_obj.get("scheme")
            identifier = id_obj.get("identifier")
            row[f"{scheme}_id"] = identifier

        rows.append(row)

All individual records are now consolidated into a single pandas DataFrame.
Each row corresponds to one legislator, and each column represents a field or identifier.

In [9]:
# Create DataFrame
df = pd.DataFrame(rows)

## **Add Missing Columns for Unused Schemes**

Some identifier schemes may not appear for all legislators.
This step ensures that **all scheme columns exist**, even if they contain only None values for some legislators.

In [10]:
# Fill in missing columns for any scheme not found in every legislator
for scheme in all_schemes:
    col_name = f"{scheme}_id"
    if col_name not in df.columns:
        df[col_name] = None

In [11]:
df.columns
df.head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,house_history_id,icpsr_id,maplight_id,opensecrets_id,pictorial_id,thomas_id,votesmart_id,wikidata_id,wikipedia_id,lis_id
0,ocd-person/4015098b-d823-5776-9b47-ceb09bf523a9,Brad Sherman,Brad,Sherman,1954-10-24,Male,https://sherman.house.gov/help,https://unitedstates.github.io/images/congress...,Democratic,lower,...,21565,29707,462,N00006897,12946,01526,142,Q672919,NaN,NaN
1,ocd-person/4c77555b-f35d-53ca-8c7d-7950b90a1b60,Dan Goldman,Dan,Goldman,1976-02-26,Male,https://goldman.house.gov/address_authenticati...,https://unitedstates.github.io/images/congress...,Democratic,lower,...,NaN,NaN,NaN,N00051424,13178,NaN,208627,Q74849175,Dan Goldman,NaN
2,ocd-person/befb976c-1369-5d86-a53f-4aeebdf486cc,Jim Banks,Jim,Banks,1979-07-16,Male,https://www.banks.senate.gov/contact/share-you...,https://unitedstates.github.io/images/congress...,Republican,lower,...,NaN,21702,2214,N00037185,13052,NaN,116801,Q16146788,NaN,NaN
3,ocd-person/ea06a401-ba19-4028-8986-38e950a657b6,Julie Johnson,Julie,Johnson,1966-05-02,Female,https://juliejohnson.house.gov/address_authent...,None,Democratic,lower,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ocd-person/a516a327-22fc-57af-931b-f28926b44b85,Rob Menendez,Rob,Menendez,1985-07-12,Male,https://menendez.house.gov/address_authenticat...,https://unitedstates.github.io/images/congress...,Democratic,lower,...,NaN,NaN,NaN,N00049686,13157,NaN,205678,Q115137457,Rob Menendez,NaN


To verify data completeness, we check if any legislators lack a bioguide_id.
The resulting DataFrame displays those records for review.

In [12]:
# are there any missing bioguide ids?
df[df['bioguide_id'].isna()].head()

,id,name,given_name,family_name,birth_date,gender,email,image,party,role_type,...,house_history_id,icpsr_id,maplight_id,opensecrets_id,pictorial_id,thomas_id,votesmart_id,wikidata_id,wikipedia_id,lis_id


## **Export to CSV**

Finally, we export the full DataFrame to a CSV file named `plural_current_legislators_with_bioguide.csv`, keeping all columns intact and disabling the index column.

In [13]:
# Define the project root directory (Google Drive)
drive_root = "/content/drive/Shareddrives/Data Archive/Bridge_Grades"

# Define the relative output path from the project root
base_dir = os.path.join(
    drive_root,
    "Input_Files",
    "Snapshot_Raw_2026_06_02",
    "Raw_Others_2026_06_02"
)

# Check if the directory exists (do not create it)
if not os.path.isdir(base_dir):
    raise OSError(f"Directory does not exist: {base_dir}")

# Define the output file name and full path
output_filename = "plural_current_legislators_with_bioguide.csv"
output_path = os.path.join(base_dir, output_filename)

# Save the DataFrame to CSV
df.to_csv(output_path, index=False)

# Display confirmation message
print(f"File saved as: {output_path}")

File saved as: /content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/plural_current_legislators_with_bioguide.csv
